In [1]:
import datetime
import json
from pathlib import Path

import cv2
from tqdm import tqdm
from insightface.app import FaceAnalysis
import onnxruntime as ort

BASE_DIR = Path("/app")
TESTSET_PATH = BASE_DIR / "testsets/four-people_testset.json"
OUTPUT_ROOT = BASE_DIR / "image_outputs"

CLASSIFIER_NAME = "insightface"
DET_SIZE = (640, 640)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = OUTPUT_ROOT / f"{timestamp}_{CLASSIFIER_NAME}_only-faceDetector_det{DET_SIZE[0]}x{DET_SIZE[1]}"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Saving annotated outputs to: {output_dir}")
print(f"Testset: {TESTSET_PATH}")

/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Saving annotated outputs to: /app/image_outputs/20251227_123104_insightface_only-faceDetector_det640x640
Testset: /app/testsets/four-people_testset.json


In [2]:
def init_insightface():
    available = ort.get_available_providers()
    prefer_cuda = "CUDAExecutionProvider" in available
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if prefer_cuda else ["CPUExecutionProvider"]
    ctx_id = 0 if prefer_cuda else -1
    try:
        app = FaceAnalysis(allowed_modules=["detection"], providers=providers)
        app.prepare(ctx_id=ctx_id, det_size=DET_SIZE)
        print(f"InsightFace initialized with providers={providers}, ctx_id={ctx_id}")
        return app
    except Exception as e:
        if prefer_cuda:
            print(f"GPU init failed ({e}); retrying on CPU only.")
            providers = ["CPUExecutionProvider"]
            ctx_id = -1
            app = FaceAnalysis(allowed_modules=["detection"], providers=providers)
            app.prepare(ctx_id=ctx_id, det_size=DET_SIZE)
            print(f"InsightFace initialized with providers={providers}, ctx_id={ctx_id}")
            return app
        raise

app = init_insightface()

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
model ignore: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvid

In [3]:
with open(TESTSET_PATH, "r", encoding="utf-8") as f:
    test_items = json.load(f)

print(f"Loaded {len(test_items)} images from test set")

Loaded 2503 images from test set


In [4]:
def annotate_and_save(item):
    img_path = BASE_DIR / item.get("path", "")
    if not img_path.exists():
        return {"path": str(img_path), "reason": "missing file"}

    img = cv2.imread(str(img_path))
    if img is None:
        return {"path": str(img_path), "reason": "unable to load"}

    faces = app.get(img)
    annotated = img.copy()

    if not faces:
        cv2.putText(annotated, "No faces detected", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    for face in faces:
        x1, y1, x2, y2 = [int(v) for v in face.bbox]
        score = getattr(face, "det_score", None)
        label = "Face" if score is None else f"Face ({score:.2f})"
        cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(annotated, label, (x1, max(y1 - 10, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    out_path = output_dir / img_path.name
    cv2.imwrite(str(out_path), annotated)
    return None

failed = []
for item in tqdm(test_items, desc="Processing images"):
    res = annotate_and_save(item)
    if res:
        failed.append(res)

print(f"Annotated images saved to {output_dir}")
if failed:
    print("Skipped images:")
    for miss in failed:
        print(miss)

Processing images: 100%|██████████| 2503/2503 [04:01<00:00, 10.38it/s]

Annotated images saved to /app/image_outputs/20251227_123104_insightface_only-faceDetector_det640x640


In [5]:
sample_outputs = sorted(output_dir.iterdir())[:5] if output_dir.exists() else []
print("Sample outputs:")
for p in sample_outputs:
    print(p)

Sample outputs:
/app/image_outputs/20251227_123104_insightface_only-faceDetector_det640x640/001_9adc92c2.jpg
/app/image_outputs/20251227_123104_insightface_only-faceDetector_det640x640/002_85eab275.jpg
/app/image_outputs/20251227_123104_insightface_only-faceDetector_det640x640/003_8889ec2c.jpg
/app/image_outputs/20251227_123104_insightface_only-faceDetector_det640x640/004_41caa173.jpg
/app/image_outputs/20251227_123104_insightface_only-faceDetector_det640x640/005_3ba56da0.jpg
